# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets (datasets):\n")

for record_set in record_sets:
    print(f"- Record Set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}\n      Name: {field.name} | Type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose target record set(s) by their @id
# For FAIR^2 (see overview output), typically the main tabular dataset is the primary record set.
# For illustration, we'll extract all record sets present
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n---\n")
    else:
        print(f"No records found for {record_set_id}")

# For subsequent steps, pick the main record set ID (edit as needed, using previous cell's output)
# If only one record set exists, use it; otherwise, set an appropriate ID. Example below (update if different):
if record_set_ids:
    main_record_set_id = record_set_ids[0]  # You may set another if multiple exist
    df = dataframes[main_record_set_id]
else:
    main_record_set_id = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes:
- Removing/inspecting outliers,
- Transforming specific data columns,
- Grouping by categorical attributes.

In [ ]:
if df is not None:
    # Try to find numeric fields for analysis
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields available: {numeric_cols}")
    
    # If none found, try to coerce common numeric columns (example columns: 'MW', 'Abundance', etc.)
    if not numeric_cols:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        print(f"Detected numeric columns after coercion: {numeric_cols}")
    
    # Pick a numeric field if available
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Update if you wish to select another field
        print(f"Analyzing field: {numeric_field}\n")
        threshold = df[numeric_field].mean()  # For demonstration, use the mean as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows.")
        display_cols = [numeric_field]
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score):")
        display_cols.append(f"{numeric_field}_normalized")
        print(filtered_df[display_cols].head())
        # Try to group by a categorical/label field
        candidate_group_fields = [
            col for col in df.columns if df[col].dtype == 'object' and col != numeric_field
        ]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping by field: {group_field}\n")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")
else:
    print("No main DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # If there are at least 2 numeric fields, plot scatter
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f"Scatter plot: {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we have:
- Loaded Croissant metadata and records from the FAIR² compliant mass spectrometry dataset.
- Inspected and extracted available record sets and fields using entity `@id` values.
- Performed basic filtering, normalization, and grouping on numeric fields for exploratory data analysis.
- Generated basic plots to visualize data distributions.

This approach can be adapted to other Croissant datasets by updating the schema URL and adjusting field selection as needed.